In [3]:
from pathlib import Path
import os

print("Jupyter kjører fra:")
print(Path.cwd())

print("\nDB_PATH:")
print(DB_PATH)

print("\nAbsolutt DB_PATH:")
print(Path(DB_PATH).resolve())

print("\nFinnes databasen?")
print(Path(DB_PATH).exists())

Jupyter kjører fra:
c:\Users\frank\Desktop\AI_KOM\kommune-ai-guardrails

DB_PATH:
kommune-ai-guardrails/data/db/kommune_rag_store.db

Absolutt DB_PATH:
C:\Users\frank\Desktop\AI_KOM\kommune-ai-guardrails\kommune-ai-guardrails\data\db\kommune_rag_store.db

Finnes databasen?
False


In [4]:
from pathlib import Path
import sqlite3
import sqlite_vec

DB_PATH = Path(
    r"C:\Users\frank\Desktop\AI_KOM\kommune-ai-guardrails\data\db\kommune_rag_store.db"
)

if not DB_PATH.is_file():
    raise FileNotFoundError(f"Databasen ble ikke funnet: {DB_PATH}")

conn = sqlite3.connect(str(DB_PATH))

conn.enable_load_extension(True)
sqlite_vec.load(conn)
conn.enable_load_extension(False)

print("✅ Database åpnet")
print("Database:", DB_PATH)
print("SQLite:", sqlite3.sqlite_version)
print("sqlite-vec:", conn.execute("SELECT vec_version()").fetchone()[0])

✅ Database åpnet
Database: C:\Users\frank\Desktop\AI_KOM\kommune-ai-guardrails\data\db\kommune_rag_store.db
SQLite: 3.50.4
sqlite-vec: v0.1.9


In [5]:
tables = conn.execute("""
    SELECT name, type
    FROM sqlite_master
    WHERE type IN ('table', 'view')
    ORDER BY name
""").fetchall()

print("\nTabeller og views:\n")

for name, object_type in tables:
    print(f"{object_type:8} {name}")


Tabeller og views:

table    Audit_AI_Actions
table    legal_chunks
table    legal_fts
table    legal_fts_config
table    legal_fts_content
table    legal_fts_data
table    legal_fts_docsize
table    legal_fts_idx
table    sqlite_sequence
table    vec_legal_chunks
table    vec_legal_chunks_chunks
table    vec_legal_chunks_info
table    vec_legal_chunks_rowids
table    vec_legal_chunks_vector_chunks00


In [6]:
vector_tables = conn.execute("""
    SELECT name, sql
    FROM sqlite_master
    WHERE sql LIKE '%USING vec0%'
""").fetchall()

print("\nVector-tabeller:\n")

for name, sql in vector_tables:
    print(f"Navn: {name}")
    print(f"SQL:  {sql}")
    print("-" * 60)


Vector-tabeller:

Navn: vec_legal_chunks
SQL:  CREATE VIRTUAL TABLE vec_legal_chunks USING vec0(
        chunk_id INTEGER PRIMARY KEY,
        embedding float[384]
    )
------------------------------------------------------------


In [7]:
print("=== legal_chunks ===")
for row in conn.execute("PRAGMA table_info(legal_chunks)").fetchall():
    print(row)

print("\n=== vec_legal_chunks ===")

schema = conn.execute("""
SELECT sql
FROM sqlite_master
WHERE name = 'vec_legal_chunks'
""").fetchone()

print(schema[0] if schema else "Fant ikke schema")


=== legal_chunks ===
(0, 'id', 'INTEGER', 0, None, 1)
(1, 'source_file', 'TEXT', 1, None, 0)
(2, 'category', 'TEXT', 1, None, 0)
(3, 'section_ref', 'TEXT', 0, None, 0)
(4, 'title', 'TEXT', 1, None, 0)
(5, 'content', 'TEXT', 1, None, 0)

=== vec_legal_chunks ===
CREATE VIRTUAL TABLE vec_legal_chunks USING vec0(
        chunk_id INTEGER PRIMARY KEY,
        embedding float[384]
    )


In [8]:
import numpy as np


def vector_search(query: str, top_k: int = 5):
    """
    Søker i vec_legal_chunks og henter tilhørende
    lovtekst fra legal_chunks.
    """

    # 1. Lag embedding av spørsmålet
    query_embedding = embedding_model.encode(
        f"query: {query}",
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32
    )

    # Kontroll
    print(f"Embedding dimensions: {len(query_embedding)}")

    # 2. Vector search
    vector_rows = conn.execute("""
        SELECT
            chunk_id,
            distance
        FROM vec_legal_chunks
        WHERE embedding MATCH ?
        ORDER BY distance
        LIMIT ?
    """, (
        query_embedding,
        top_k
    )).fetchall()

    # 3. Hent originaltekst
    results = []

    for chunk_id, distance in vector_rows:

        chunk = conn.execute("""
            SELECT
                id,
                source_file,
                category,
                section_ref,
                title,
                content
            FROM legal_chunks
            WHERE id = ?
        """, (chunk_id,)).fetchone()

        if chunk:

            results.append({
                "id": chunk[0],
                "distance": distance,
                "source_file": chunk[1],
                "category": chunk[2],
                "section_ref": chunk[3],
                "title": chunk[4],
                "content": chunk[5]
            })

    return results


# ==========================================
# TEST
# ==========================================

query = (
    "Hvilke regler gjelder dersom vi må "
    "endre budsjettet i løpet av året?"
)

results = vector_search(
    query,
    top_k=5
)


# ==========================================
# VIS RESULTATER
# ==========================================

print("\nSPØRSMÅL:")
print(query)

for i, result in enumerate(results, start=1):

    print("\n" + "=" * 80)
    print(f"RESULTAT #{i}")
    print("=" * 80)

    print(f"Chunk ID : {result['id']}")
    print(f"Distance : {result['distance']:.4f}")
    print(f"Kilde    : {result['source_file']}")
    print(f"Kategori : {result['category']}")
    print(f"Paragraf : {result['section_ref']}")
    print(f"Tittel   : {result['title']}")

    print("\nTEKST:")
    print("-" * 80)
    print(result["content"])

Embedding dimensions: 384

SPØRSMÅL:
Hvilke regler gjelder dersom vi må endre budsjettet i løpet av året?

RESULTAT #1
Chunk ID : 1683
Distance : 0.4842
Kilde    : Lov om kommuner og fylkeskommuner (kommuneloven) - Lovdata.pdf
Kategori : Kommuneloven
Paragraf : § 14-5
Tittel   : § 14-5 Årsbudsjettets bindende virkning

TEKST:
--------------------------------------------------------------------------------
§ 14-5.Årsbudsjettets bindende virkning. Budsjettstyring
Årsbudsjettet er bindende for kommunestyret, fylkestinget og underordnete organer. Første punktum er ikke
til hinder for at kommunestyret eller fylkestinget kan gi et underordnet organ myndighet til å avgjøre at deler av
en bevilgning i driftsbudsjettet skal benyttes til å finansiere utgifter i investeringsregnskapet. Første punktum
gjelder ikke for utbetalinger som kommunen eller fylkeskommunen har en rettslig plikt til å foreta.
Kommunestyret eller fylkestinget skal endre årsbudsjettet når det er nødvendig for å oppfylle loven